# 02 — Exploratory Data Analysis

This notebook analyzes the merged dataset created by
`01_extract_and_merge_cr_fiqa_scores.ipynb`.

The analysis includes:

- descriptive CR-FIQA statistics,
- score distributions,
- correlations with continuous and binary facial attributes,
- demographic group distributions,
- CR-FIQA comparisons across demographic groups,
- selected scatterplots,
- export of figures, tables, and a text summary.

## Execution order

1. `00_colab_setup.ipynb`
2. `01_extract_and_merge_cr_fiqa_scores.ipynb`
3. `02_exploratory_data_analysis.ipynb`

Keep all three notebooks in the same `notebooks/` folder.


In [ ]:
# Locate and run the shared setup notebook.
#
# Recommended repository layout:
# fiqa-demographic-analysis/
# └── notebooks/
#     ├── 00_colab_setup.ipynb
#     ├── 01_extract_and_merge_cr_fiqa_scores.ipynb
#     └── 02_exploratory_data_analysis.ipynb

from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(
        f"- {path}" for path in setup_candidates
    )
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep the notebooks in the same notebooks/ folder or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")

get_ipython().run_line_magic(
    "run",
    f'"{SETUP_NOTEBOOK}"',
)


In [ ]:
# Imports and output paths

import matplotlib.pyplot as plt
import pandas as pd

MERGED_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

RESULTS_PATH = PROJECT_PATH / "results" / "eda"
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"

FIGURES_PATH.mkdir(parents=True, exist_ok=True)
TABLES_PATH.mkdir(parents=True, exist_ok=True)

if not MERGED_FILE.exists():
    raise FileNotFoundError(
        f"Merged dataset was not found: {MERGED_FILE}\n"
        "Run 01_extract_and_merge_cr_fiqa_scores.ipynb first."
    )

print(f"Merged dataset: {MERGED_FILE}")
print(f"EDA results:    {RESULTS_PATH}")


In [ ]:
# Load and validate the merged dataset

df = pd.read_csv(MERGED_FILE)

required_columns = {
    "index",
    "cr_fiqa_score",
    "group",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise KeyError(
        "The following required columns are missing: "
        f"{sorted(missing_columns)}"
    )

if df.empty:
    raise RuntimeError("The merged dataset contains no rows.")

if not pd.api.types.is_numeric_dtype(df["cr_fiqa_score"]):
    raise TypeError("'cr_fiqa_score' must be numeric.")

duplicate_indices = int(df["index"].duplicated().sum())
missing_values = df.isna().sum().sort_values(ascending=False)

print(f"Dataset shape: {df.shape}")
print(f"Duplicate image indices: {duplicate_indices}")

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(missing_values.to_frame("missing_values"))


In [ ]:
# Descriptive CR-FIQA statistics

fiqa_statistics = df["cr_fiqa_score"].describe()

print("CR-FIQA score statistics:")
display(fiqa_statistics.to_frame("cr_fiqa_score"))

fiqa_statistics.to_csv(
    TABLES_PATH / "fiqa_score_statistics.csv",
    header=["cr_fiqa_score"],
)


In [ ]:
# CR-FIQA score distribution

plt.figure(figsize=(8, 5))
plt.hist(
    df["cr_fiqa_score"].dropna(),
    bins=40,
)
plt.xlabel("CR-FIQA Score")
plt.ylabel("Number of Images")
plt.title("Distribution of CR-FIQA Scores")
plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "fiqa_score_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Overall CR-FIQA score boxplot

plt.figure(figsize=(6, 5))
plt.boxplot(
    df["cr_fiqa_score"].dropna(),
    vert=True,
)
plt.ylabel("CR-FIQA Score")
plt.title("Boxplot of CR-FIQA Scores")
plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "fiqa_score_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Define feature types
#
# Continuous, binary, and categorical variables are kept separate so
# categorical group labels are not treated as continuous measurements.

continuous_features = [
    "age",
    "smile",
    "moustache",
    "beard",
    "sideburns",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

binary_features = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

categorical_features = [
    "group",
]

continuous_features = [
    feature
    for feature in continuous_features
    if feature in df.columns
]

binary_features = [
    feature
    for feature in binary_features
    if feature in df.columns
]

categorical_features = [
    feature
    for feature in categorical_features
    if feature in df.columns
]

print("Continuous features:")
print(continuous_features)

print("\nBinary features:")
print(binary_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:
# Correlation analysis for continuous and binary features

correlation_features = continuous_features + binary_features

if not correlation_features:
    raise RuntimeError(
        "No continuous or binary features are available for correlation analysis."
    )

correlations = (
    df[correlation_features + ["cr_fiqa_score"]]
    .corr(numeric_only=True)["cr_fiqa_score"]
    .drop("cr_fiqa_score")
    .dropna()
    .sort_values(
        key=abs,
        ascending=False,
    )
)

correlation_table = correlations.to_frame(
    "correlation_with_fiqa"
)

print("Feature correlations with CR-FIQA:")
display(correlation_table)

correlation_table.to_csv(
    TABLES_PATH / "correlations_with_fiqa.csv"
)


In [ ]:
# Plot the strongest correlations

number_of_features = min(15, len(correlations))

if number_of_features > 0:
    top_correlations = correlations.head(number_of_features)

    plt.figure(figsize=(9, 6))
    plt.barh(
        top_correlations.index[::-1],
        top_correlations.values[::-1],
    )
    plt.axvline(x=0, linewidth=1)
    plt.xlabel("Correlation with CR-FIQA Score")
    plt.title(
        f"Top {number_of_features} Attribute Correlations "
        "with CR-FIQA"
    )
    plt.tight_layout()

    plt.savefig(
        FIGURES_PATH / "top_correlations_with_fiqa.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("No valid correlations were available for plotting.")


In [ ]:
# Demographic group distribution

group_counts = df["group"].value_counts(dropna=False)

print("Number of images per demographic group:")
display(group_counts.to_frame("count"))

group_counts.to_csv(
    TABLES_PATH / "group_counts.csv",
    header=["count"],
)

plt.figure(figsize=(9, 5))
plt.bar(
    group_counts.index.astype(str),
    group_counts.values,
)
plt.xlabel("Demographic Group")
plt.ylabel("Number of Images")
plt.title("Number of Images per Demographic Group")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "group_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# CR-FIQA statistics by demographic group

group_statistics = (
    df.groupby(
        "group",
        dropna=False,
    )["cr_fiqa_score"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max",
    )
    .sort_values(
        "mean",
        ascending=False,
    )
)

print("CR-FIQA statistics by demographic group:")
display(group_statistics)

group_statistics.to_csv(
    TABLES_PATH / "fiqa_statistics_by_group.csv"
)


In [ ]:
# Mean CR-FIQA score by demographic group

plt.figure(figsize=(9, 5))
plt.bar(
    group_statistics.index.astype(str),
    group_statistics["mean"],
)
plt.xlabel("Demographic Group")
plt.ylabel("Mean CR-FIQA Score")
plt.title("Mean CR-FIQA Score by Demographic Group")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "mean_fiqa_by_group.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
# CR-FIQA score distribution by demographic group

group_names = group_statistics.index.tolist()

scores_by_group = []
group_labels = []

for group in group_names:
    if pd.isna(group):
        group_scores = df.loc[
            df["group"].isna(),
            "cr_fiqa_score",
        ].dropna()
        group_label = "Missing"
    else:
        group_scores = df.loc[
            df["group"] == group,
            "cr_fiqa_score",
        ].dropna()
        group_label = str(group)

    if not group_scores.empty:
        scores_by_group.append(group_scores)
        group_labels.append(group_label)

if scores_by_group:
    plt.figure(figsize=(10, 6))
    plt.boxplot(
        scores_by_group,
        labels=group_labels,
    )
    plt.xlabel("Demographic Group")
    plt.ylabel("CR-FIQA Score")
    plt.title(
        "CR-FIQA Score Distribution by Demographic Group"
    )
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    plt.savefig(
        FIGURES_PATH / "fiqa_boxplot_by_group.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("No demographic groups were available for the boxplot.")


In [ ]:
# Mean feature values by demographic group
#
# For binary variables, the mean represents the share of images for
# which the attribute is present.

attributes_for_group_comparison = (
    continuous_features + binary_features
)

attribute_group_means = df.groupby(
    "group",
    dropna=False,
)[attributes_for_group_comparison].mean()

print("Mean attribute values by demographic group:")
display(attribute_group_means)

attribute_group_means.to_csv(
    TABLES_PATH / "attribute_means_by_group.csv"
)


In [ ]:
# Scatterplots for selected continuous features

scatterplot_features = [
    "exposure",
    "smile",
    "head_yaw",
    "blur",
]

scatterplot_features = [
    feature
    for feature in scatterplot_features
    if feature in continuous_features
]

for feature in scatterplot_features:
    plot_data = df[
        [feature, "cr_fiqa_score"]
    ].dropna()

    if len(plot_data) < 2:
        print(
            f"Skipping {feature}: fewer than two complete observations."
        )
        continue

    correlation = plot_data[
        [feature, "cr_fiqa_score"]
    ].corr().iloc[0, 1]

    plt.figure(figsize=(8, 6))
    plt.scatter(
        plot_data[feature],
        plot_data["cr_fiqa_score"],
        alpha=0.25,
        s=10,
    )
    plt.xlabel(feature.replace("_", " ").title())
    plt.ylabel("CR-FIQA Score")
    plt.title(
        f"{feature.replace('_', ' ').title()} "
        "vs. CR-FIQA Score\n"
        f"Pearson correlation = {correlation:.3f}"
    )
    plt.tight_layout()

    plt.savefig(
        FIGURES_PATH / f"{feature}_vs_fiqa.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
# Save the EDA text summary

SUMMARY_FILE = RESULTS_PATH / "eda_summary.txt"

with SUMMARY_FILE.open(
    mode="w",
    encoding="utf-8",
) as summary_file:
    summary_file.write(
        "Exploratory Data Analysis Summary\n"
    )
    summary_file.write("=" * 60 + "\n\n")

    summary_file.write(
        f"Dataset shape: {df.shape}\n"
    )
    summary_file.write(
        f"Duplicate image indices: {duplicate_indices}\n\n"
    )

    summary_file.write(
        "CR-FIQA score statistics\n"
    )
    summary_file.write("-" * 60 + "\n")
    summary_file.write(fiqa_statistics.to_string())

    summary_file.write(
        "\n\nMissing values\n"
    )
    summary_file.write("-" * 60 + "\n")
    summary_file.write(missing_values.to_string())

    summary_file.write(
        "\n\nCorrelations with CR-FIQA\n"
    )
    summary_file.write("-" * 60 + "\n")
    summary_file.write(correlations.to_string())

    summary_file.write(
        "\n\nCR-FIQA statistics by demographic group\n"
    )
    summary_file.write("-" * 60 + "\n")
    summary_file.write(group_statistics.to_string())

print("EDA completed successfully.")
print(f"Figures saved to: {FIGURES_PATH}")
print(f"Tables saved to:  {TABLES_PATH}")
print(f"Summary saved to: {SUMMARY_FILE}")
